# SPRKD Data Analysis Notebook

Consolidates the analyses from the original `SPRKD_DATA_ANALYSIS_AND_HESSIAN_VISUALIZATIONS.ipynb` against the bundled LFS artifacts:

1. Loss / accuracy curves (paper Figure 2) from `METRICS/LOSSES AND ACCURACIES/`
2. Hessian Eigenvalue Spectral Density (ESD) (paper Figure 3) from `METRICS/HESSIAN EIGENSPECTRA/`
3. McNemar paired statistical test (paper Table 1) on `TESTSET.pth`
4. 2-D loss-landscape sweep (paper Figure 4) for the released SPRKD student

All cells run end-to-end on CPU / CUDA / Apple Silicon (MPS).

In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sprkd import (
    load_legacy_metrics_pkl,
    load_legacy_student,
    pairwise_mcnemar,
)
from sprkd.data import load_testset_pth
from sprkd.legacy import epoch_validation_series
from sprkd.utils import get_device

REPO = Path('..').resolve()
device = get_device()
print('device:', device)

## 1. Loss / accuracy curves (paper Figure 2)

The released metric pickles store dicts of the form `{'TRAINING': {...}, 'VALIDATION': {...}}` with per-step losses and accuracies.

In [ ]:
metrics_dir = REPO / 'METRICS' / 'LOSSES AND ACCURACIES'
candidates = {
    'SPRKD':   metrics_dir / '500_SPRKD_LOSSES.pkl',
    'Control': metrics_dir / '500_CONTROL_STUDENT_LOSSES.pkl',
    'RKD':     metrics_dir / 'RKD_STUDENT_METRICS.pkl',
}

loaded = {}
for label, path in candidates.items():
    if not path.is_file():
        print(f'  [skip] {label}: {path.name} missing')
        continue
    loaded[label] = load_legacy_metrics_pkl(path)
    n = len(loaded[label].get('VALIDATION', {}).get('LOSSES', []))
    print(f'  [ok]   {label}: {n} validation steps')
    losses, accs = epoch_validation_series(loaded[label], step_size=323)
    if accs:
        print(f'         final epoch acc (step-323): {accs[-1]:.2f}%')

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4.5))
for label, hist in loaded.items():
    v_losses = [float(x) for x in hist['VALIDATION']['LOSSES']]
    v_accs   = [float(x) for x in hist['VALIDATION']['ACCURACIES']]
    ax_loss.plot(v_losses, label=label, alpha=0.85)
    ax_acc.plot(v_accs, label=label, alpha=0.85)
ax_loss.set_xlabel('validation step'); ax_loss.set_ylabel('loss'); ax_loss.set_title('Validation loss')
ax_acc.set_xlabel('validation step');  ax_acc.set_ylabel('acc (%)'); ax_acc.set_title('Validation accuracy')
for ax in (ax_loss, ax_acc): ax.grid(True, alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig('analysis_loss_acc.png', dpi=120)
print('saved analysis_loss_acc.png')

## 2. Hessian Eigenvalue Spectral Density (paper Figure 3)

Each saved eigenspectrum file holds the PyHessian `(eigenvalues, weights)` tuple, possibly with a Hessian trace appended.

In [ ]:
from sprkd.analysis import density_to_histogram

esd_dir = REPO / 'METRICS' / 'HESSIAN EIGENSPECTRA'
esd_candidates = {
    'SPRKD':   esd_dir / 'EIGS_500_SPRKD_MALARIA.pth',
    'Control': esd_dir / 'EIGS_500_CONTROL_MALARIA.pth',
    'RKD':     esd_dir / 'EIGS_RKD_MALARIA_STUDENT.pth',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (label, path) in zip(axes, esd_candidates.items()):
    if not path.is_file():
        ax.set_title(f'{label} (missing)')
        continue
    payload = torch.load(path, map_location='cpu', weights_only=False)
    trace = None
    if isinstance(payload, (list, tuple)) and len(payload) >= 2:
        eigenvalues, weights = payload[0], payload[1]
    elif isinstance(payload, dict):
        eigenvalues = (
            payload.get('eigenvalues')
            or payload.get('eigenvalue')
            or payload.get('EIG_DENSITY')
        )
        weights = (
            payload.get('weights')
            or payload.get('weight')
            or payload.get('WEIGHT_DENSITY')
        )
        trace = payload.get('TRACE')
        # bundled ISEF artifacts wrap density samples in a length-1 list
        if isinstance(eigenvalues, list) and len(eigenvalues) == 1 and isinstance(eigenvalues[0], list):
            eigenvalues = eigenvalues[0]
        if isinstance(weights, list) and len(weights) == 1 and isinstance(weights[0], list):
            weights = weights[0]
    else:
        ax.set_title(f'{label}: unsupported format'); continue
    grid, density = density_to_histogram(eigenvalues, weights, bins=200, sigma_squared=1e-5)
    ax.plot(grid, density)
    ax.set_yscale('log'); ax.set_xlabel(r'$\lambda$'); ax.set_ylabel('density')
    title = f'{label}' + (f' (trace={float(trace):.2f})' if trace is not None else '')
    ax.set_title(title); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig('analysis_esd.png', dpi=120)
print('saved analysis_esd.png')

## 3. McNemar paired statistical test (paper Table 1)

`TESTSET.pth` is the held-out batch the paper uses for its McNemar entries (see Section 5, p-values like `6.3 x 10^-87`). With only 100 samples here we cannot reach those p-value magnitudes - the paper's matrix is computed on the entire 6,890-sample validation split. We still report the table as a sanity check.

In [ ]:
MODELS = {
    'SPRKD':   REPO / 'MODELS' / 'SPRKD_MALARIA.pth',
    'Control': REPO / 'MODELS' / 'CONTROL_MALARIA.pth',
    'RKD':     REPO / 'MODELS' / 'RKD_MALARIA_STUDENT.pth',
}

xs, ys = load_testset_pth(REPO / 'TESTSET.pth')
xs, ys = xs.to(device), ys.to(device)

preds = {}
for label, path in MODELS.items():
    if not path.is_file():
        print(f'  [skip] {label}: {path.name} missing'); continue
    m = load_legacy_student(path).to(device).eval()
    with torch.no_grad():
        preds[label] = torch.argmax(m(xs), dim=1).cpu()

rows = pairwise_mcnemar(preds, ys.cpu(), method='exact')
for r in rows:
    print(
        f"  {r['model_a']:<8} vs {r['model_b']:<8}  "
        f"a={r['a_correct_b_correct']:>3}  b={r['a_correct_b_wrong']:>3}  "
        f"c={r['a_wrong_b_correct']:>3}  d={r['a_wrong_b_wrong']:>3}  "
        f"p = {r['p_value']:.4f}"
    )

## 4. 2-D loss-landscape sweep (paper Figure 4)

Sweeps `theta + alpha * v_1 + beta * v_2` for the SPRKD student over a small grid of perturbations along the top-2 Hessian eigenvectors of the model on `TESTSET.pth`. We use a small grid (15x15) to stay under a minute.

In [ ]:
from sprkd.landscape import compute_landscape
from sprkd.visualize import plot_loss_landscape

model = load_legacy_student(REPO / 'MODELS' / 'SPRKD_MALARIA.pth').to(device)
model.eval()

criterion = nn.CrossEntropyLoss()
# Use a small slice of TESTSET so the Hessian-vector products are fast.
small = (xs[:32].to(device), ys[:32].to(device))

landscape = compute_landscape(
    model, criterion, small,
    grid_size=11,
    progress=False,
)
print('eigenvalues:', landscape.eigenvalues)
print('losses range: [%.4f, %.4f]' % (float(np.min(landscape.losses)), float(np.max(landscape.losses))))

fig = plot_loss_landscape(landscape, title='SPRKD student', save_path='analysis_landscape.png')
print('saved analysis_landscape.png')